<a href="https://colab.research.google.com/github/VinayaSharada/FinancialAnalyticsCourse/blob/main/CreditCardFraud/credit_card_fraud_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Credit Card Fraud Detection with Machine Learning

This notebook demonstrates comprehensive credit card fraud detection using various machine learning algorithms with extensive analysis and visualization.

**Course:** Masters in Finance - AI/ML Applications  
**Topic:** Credit Card Fraud Detection  
**Objective:** Build and evaluate ML models for detecting fraudulent credit card transactions

## 📦 Setup and Installation

Install required packages for machine learning and data analysis.

In [ ]:
# Install required packages
!pip install pandas numpy scikit-learn matplotlib seaborn plotly imbalanced-learn -q

# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Machine Learning imports
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.metrics import (classification_report, confusion_matrix, roc_auc_score, 
                           roc_curve, precision_recall_curve, f1_score, accuracy_score)
from sklearn.decomposition import PCA
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

import os
from datetime import datetime

# Set plotting style
plt.style.use('default')
sns.set_palette("husl")

print("✅ All packages installed and imported successfully!")

## 📥 Load Data

Load the credit card transaction data. This notebook can work with:
1. Synthetic data generated by our data generator
2. Data loaded from GitHub repository

In [ ]:
# 📊 DATA LOADING CONFIGURATION

# Option 1: Load from GitHub (recommended for Colab)
GITHUB_URL = "https://raw.githubusercontent.com/VinayaSharada/FinancialAnalyticsCourse/main/CreditCardFraud/credit_card_transactions.csv"

# Option 2: Local file (if running locally)
LOCAL_FILE = "credit_card_transactions.csv"

try:
    # Try loading from GitHub first
    print("🔄 Loading data from GitHub repository...")
    df = pd.read_csv(GITHUB_URL)
    print(f"✅ Data loaded successfully from GitHub!")
except:
    try:
        # Fallback to local file
        print("🔄 Loading data from local file...")
        df = pd.read_csv(LOCAL_FILE)
        print(f"✅ Data loaded successfully from local file!")
    except:
        print("❌ Could not load data. Please ensure the data file exists.")
        print("💡 You can generate synthetic data using the data generator notebook first.")
        raise

# Convert timestamp to datetime
if 'timestamp' in df.columns:
    df['timestamp'] = pd.to_datetime(df['timestamp'])

# Display basic information
print("\n" + "="*60)
print("📊 DATASET OVERVIEW")
print("="*60)
print(f"Dataset shape: {df.shape}")
print(f"Total transactions: {len(df):,}")
print(f"Fraudulent transactions: {df['is_fraud'].sum():,}")
print(f"Fraud rate: {df['is_fraud'].mean()*100:.3f}%")
print(f"Date range: {df['timestamp'].min()} to {df['timestamp'].max()}")
print(f"Amount range: ₹{df['amount'].min():.2f} to ₹{df['amount'].max():.2f}")
print(f"Average amount: ₹{df['amount'].mean():.2f}")

# Check for missing values
missing_values = df.isnull().sum()
if missing_values.any():
    print(f"\n⚠️ Missing values found:")
    print(missing_values[missing_values > 0])
else:
    print(f"\n✅ No missing values found")

# Display first few rows
print("\n📋 Sample Data:")
display(df.head())

## 🔍 Exploratory Data Analysis (EDA)

Comprehensive analysis of the credit card transaction data to understand patterns and relationships.

In [ ]:
# Create comprehensive EDA plots
fig, axes = plt.subplots(2, 3, figsize=(20, 12))
fig.suptitle('Credit Card Fraud Detection - Exploratory Data Analysis', fontsize=16, fontweight='bold')

# 1. Fraud Distribution
fraud_counts = df['is_fraud'].value_counts()
colors = ['lightblue', 'red']
axes[0,0].pie(fraud_counts.values, labels=['Normal', 'Fraud'], autopct='%1.2f%%', 
             colors=colors, startangle=90)
axes[0,0].set_title('Transaction Distribution', fontweight='bold')

# 2. Amount Distribution by Fraud Status
df.boxplot(column='amount', by='is_fraud', ax=axes[0,1])
axes[0,1].set_title('Amount Distribution by Fraud Status', fontweight='bold')
axes[0,1].set_xlabel('Fraud Status (0=Normal, 1=Fraud)')
axes[0,1].set_ylabel('Amount (₹)')

# 3. Fraud by Hour of Day
fraud_by_hour = df.groupby('hour_of_day')['is_fraud'].mean()
axes[0,2].bar(fraud_by_hour.index, fraud_by_hour.values, color='orange', alpha=0.7)
axes[0,2].set_title('Fraud Rate by Hour of Day', fontweight='bold')
axes[0,2].set_xlabel('Hour')
axes[0,2].set_ylabel('Fraud Rate')
axes[0,2].grid(True, alpha=0.3)

# 4. Fraud by Merchant Category
fraud_by_category = df.groupby('merchant_category')['is_fraud'].mean().sort_values(ascending=True)
axes[1,0].barh(fraud_by_category.index, fraud_by_category.values, color='green', alpha=0.7)
axes[1,0].set_title('Fraud Rate by Merchant Category', fontweight='bold')
axes[1,0].set_xlabel('Fraud Rate')

# 5. Amount vs Age Scatter (colored by fraud)
normal_sample = df[df['is_fraud'] == 0].sample(n=min(1000, len(df[df['is_fraud'] == 0])))
fraud_data = df[df['is_fraud'] == 1]

axes[1,1].scatter(normal_sample['customer_age'], normal_sample['amount'], 
                 alpha=0.6, label='Normal', s=10, color='blue')
axes[1,1].scatter(fraud_data['customer_age'], fraud_data['amount'], 
                 alpha=0.8, label='Fraud', s=15, color='red')
axes[1,1].set_xlabel('Customer Age')
axes[1,1].set_ylabel('Amount (₹)')
axes[1,1].set_title('Amount vs Age Distribution', fontweight='bold')
axes[1,1].legend()
axes[1,1].grid(True, alpha=0.3)

# 6. Correlation heatmap for numerical features
numerical_cols = ['amount', 'customer_age', 'hour_of_day', 'amount_to_income_ratio', 
                 'amount_to_limit_ratio', 'is_fraud']
correlation_matrix = df[numerical_cols].corr()

sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, 
           ax=axes[1,2], fmt='.2f', square=True)
axes[1,2].set_title('Feature Correlation Matrix', fontweight='bold')

plt.tight_layout()
plt.show()

# Print key insights
print("\n🔍 KEY INSIGHTS FROM EDA:")
print("="*50)
print(f"• Fraud rate varies significantly by merchant category")
print(f"• Highest risk category: {fraud_by_category.idxmax()} ({fraud_by_category.max()*100:.1f}%)")
print(f"• Lowest risk category: {fraud_by_category.idxmin()} ({fraud_by_category.min()*100:.1f}%)")
print(f"• Peak fraud hours: {fraud_by_hour.nlargest(3).index.tolist()}")
print(f"• Average fraud amount: ₹{df[df['is_fraud']==1]['amount'].mean():.2f}")
print(f"• Average normal amount: ₹{df[df['is_fraud']==0]['amount'].mean():.2f}")

## 📊 Interactive Analysis

Create interactive visualizations for deeper data exploration.

In [ ]:
# Interactive time series analysis
df['date'] = df['timestamp'].dt.date
daily_stats = df.groupby('date').agg({
    'is_fraud': ['count', 'sum', 'mean'],
    'amount': 'mean'
}).round(4)

daily_stats.columns = ['total_transactions', 'fraud_count', 'fraud_rate', 'avg_amount']
daily_stats = daily_stats.reset_index()

# Create subplots
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Daily Fraud Rate', 'Daily Transaction Volume', 
                  'Daily Average Amount', 'Amount Distribution by Fraud Status'),
    specs=[[{"secondary_y": False}, {"secondary_y": False}],
           [{"secondary_y": False}, {"secondary_y": False}]]
)

# Daily fraud rate
fig.add_trace(
    go.Scatter(x=daily_stats['date'], y=daily_stats['fraud_rate'],
              name='Fraud Rate', line=dict(color='red', width=2),
              hovertemplate='Date: %{x}<br>Fraud Rate: %{y:.3f}<extra></extra>'),
    row=1, col=1
)

# Daily transaction volume
fig.add_trace(
    go.Bar(x=daily_stats['date'], y=daily_stats['total_transactions'],
          name='Transaction Count', marker_color='blue'),
    row=1, col=2
)

# Daily average amount
fig.add_trace(
    go.Scatter(x=daily_stats['date'], y=daily_stats['avg_amount'],
              name='Avg Amount', line=dict(color='green', width=2),
              hovertemplate='Date: %{x}<br>Avg Amount: ₹%{y:.2f}<extra></extra>'),
    row=2, col=1
)

# Amount distribution
fig.add_trace(
    go.Histogram(x=df[df['is_fraud']==0]['amount'], 
                name='Normal', opacity=0.7, nbinsx=50, marker_color='blue'),
    row=2, col=2
)
fig.add_trace(
    go.Histogram(x=df[df['is_fraud']==1]['amount'], 
                name='Fraud', opacity=0.7, nbinsx=50, marker_color='red'),
    row=2, col=2
)

fig.update_layout(height=800, title_text="Credit Card Fraud Analysis Dashboard", showlegend=True)
fig.show()

# Interactive scatter plot
sample_df = df.sample(n=min(3000, len(df)))  # Sample for performance
fig_scatter = px.scatter(sample_df, 
                        x='customer_age', y='amount', 
                        color='is_fraud',
                        color_discrete_map={0: 'blue', 1: 'red'},
                        title='Transaction Amount vs Customer Age',
                        labels={'is_fraud': 'Fraud Status', 
                               'customer_age': 'Customer Age',
                               'amount': 'Transaction Amount (₹)'},
                        hover_data=['merchant_category', 'transaction_city', 'hour_of_day'])

fig_scatter.update_layout(height=500)
fig_scatter.show()

print("✅ Interactive visualizations created successfully!")

## 🔧 Feature Engineering and Preparation

Prepare the data for machine learning by selecting features, encoding categorical variables, and handling class imbalance.

In [ ]:
# Feature preparation parameters
TEST_SIZE = 0.2
RANDOM_STATE = 42
APPLY_SMOTE = True

print("🔧 FEATURE ENGINEERING AND DATA PREPARATION")
print("="*60)

# Select features (exclude non-predictive columns)
exclude_columns = ['transaction_id', 'customer_id', 'timestamp', 'is_fraud',
                  'customer_city', 'transaction_city', 'date']

feature_columns = [col for col in df.columns if col not in exclude_columns]
print(f"📋 Selected features: {len(feature_columns)}")
print(f"Features: {feature_columns}")

# Handle categorical variables
df_processed = df.copy()

# One-hot encode categorical variables
categorical_cols = ['merchant_category']
print(f"\n🔄 Encoding categorical variables: {categorical_cols}")

df_encoded = pd.get_dummies(df_processed, columns=categorical_cols, prefix=categorical_cols)

# Update feature columns after encoding
feature_columns = [col for col in df_encoded.columns if col not in exclude_columns]
print(f"📊 Features after encoding: {len(feature_columns)}")

# Prepare X and y
X = df_encoded[feature_columns]
y = df_encoded['is_fraud']

print(f"\n📐 Dataset dimensions:")
print(f"   • Features (X): {X.shape}")
print(f"   • Target (y): {y.shape}")
print(f"   • Fraud rate: {y.mean()*100:.3f}%")

# Split the data
print(f"\n✂️ Splitting data (test_size={TEST_SIZE})...")
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")
print(f"Training fraud rate: {y_train.mean()*100:.3f}%")
print(f"Test fraud rate: {y_test.mean()*100:.3f}%")

# Feature scaling
print(f"\n📏 Applying feature scaling...")
scaler = RobustScaler()  # Less sensitive to outliers than StandardScaler
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Handle class imbalance with SMOTE
if APPLY_SMOTE:
    print(f"\n⚖️ Applying SMOTE for class balancing...")
    smote = SMOTE(random_state=RANDOM_STATE, sampling_strategy=0.3)  # 30% fraud ratio
    X_train_balanced, y_train_balanced = smote.fit_resample(X_train_scaled, y_train)
    
    print(f"Original training samples: {len(y_train)}")
    print(f"Balanced training samples: {len(y_train_balanced)}")
    print(f"Original fraud rate: {y_train.mean()*100:.3f}%")
    print(f"Balanced fraud rate: {y_train_balanced.mean()*100:.3f}%")
else:
    X_train_balanced = X_train_scaled
    y_train_balanced = y_train

print("\n✅ Feature preparation completed successfully!")

## 🤖 Machine Learning Model Training

Train multiple machine learning models for fraud detection and compare their performance.

In [ ]:
print("🤖 MACHINE LEARNING MODEL TRAINING")
print("="*60)

# Define models to train
models = {
    'Logistic Regression': LogisticRegression(random_state=RANDOM_STATE, max_iter=1000, class_weight='balanced'),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, class_weight='balanced'),
    'Isolation Forest': IsolationForest(contamination=0.02, random_state=RANDOM_STATE)
}

# Store trained models and results
trained_models = {}
model_results = {}

# Train each model
for name, model in models.items():
    print(f"\n🔄 Training {name}...")
    
    try:
        if name == 'Isolation Forest':
            # Isolation Forest is unsupervised
            model.fit(X_train_scaled)
            
            # Predict on test set (-1 for outliers/fraud, 1 for inliers/normal)
            predictions = model.predict(X_test_scaled)
            # Convert to binary (1 for fraud, 0 for normal)
            y_pred = np.where(predictions == -1, 1, 0)
            
            # Get anomaly scores
            anomaly_scores = model.decision_function(X_test_scaled)
            # Convert scores to probabilities (higher score = less anomalous)
            y_pred_proba = 1 - ((anomaly_scores - anomaly_scores.min()) / 
                              (anomaly_scores.max() - anomaly_scores.min()))
            
        else:
            # Supervised models
            model.fit(X_train_balanced, y_train_balanced)
            y_pred = model.predict(X_test_scaled)
            y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]
        
        # Calculate metrics
        accuracy = accuracy_score(y_test, y_pred)
        precision = (y_pred * y_test).sum() / (y_pred.sum() + 1e-8)
        recall = (y_pred * y_test).sum() / (y_test.sum() + 1e-8)
        f1 = f1_score(y_test, y_pred)
        
        try:
            auc_score = roc_auc_score(y_test, y_pred_proba)
        except:
            auc_score = 0.5
        
        # Store results
        trained_models[name] = model
        model_results[name] = {
            'model': model,
            'predictions': y_pred,
            'probabilities': y_pred_proba,
            'accuracy': accuracy,
            'precision': precision,
            'recall': recall,
            'f1_score': f1,
            'auc_score': auc_score
        }
        
        print(f"✅ {name} trained successfully!")
        print(f"   • Accuracy: {accuracy:.3f}")
        print(f"   • Precision: {precision:.3f}")
        print(f"   • Recall: {recall:.3f}")
        print(f"   • F1-Score: {f1:.3f}")
        print(f"   • AUC: {auc_score:.3f}")
        
    except Exception as e:
        print(f"❌ Error training {name}: {str(e)}")
        continue

# Model comparison table
print("\n" + "="*80)
print("📊 MODEL PERFORMANCE COMPARISON")
print("="*80)
print(f"{'Model':<20} {'Accuracy':<10} {'Precision':<10} {'Recall':<10} {'F1-Score':<10} {'AUC':<10}")
print("-" * 80)

for name, result in model_results.items():
    print(f"{name:<20} {result['accuracy']:<10.3f} {result['precision']:<10.3f} "
          f"{result['recall']:<10.3f} {result['f1_score']:<10.3f} {result['auc_score']:<10.3f}")

# Find best model
best_model_name = max(model_results.keys(), key=lambda k: model_results[k]['f1_score'])
print(f"\n🏆 Best performing model: {best_model_name} (F1-Score: {model_results[best_model_name]['f1_score']:.3f})")

print("\n✅ Model training completed successfully!")

## 📈 Model Evaluation and Visualization

Comprehensive evaluation of trained models with detailed visualizations.

In [ ]:
print("📈 MODEL EVALUATION AND VISUALIZATION")
print("="*60)

# Create comprehensive evaluation plots
fig, axes = plt.subplots(2, 3, figsize=(20, 12))
fig.suptitle('Credit Card Fraud Detection - Model Evaluation Results', fontsize=16, fontweight='bold')

# 1. Model Performance Comparison
metrics = ['accuracy', 'precision', 'recall', 'f1_score', 'auc_score']
model_names = list(model_results.keys())

x = np.arange(len(model_names))
width = 0.15

for i, metric in enumerate(metrics):
    values = [model_results[name][metric] for name in model_names]
    axes[0,0].bar(x + i*width, values, width, label=metric.replace('_', ' ').title())

axes[0,0].set_xlabel('Models')
axes[0,0].set_ylabel('Score')
axes[0,0].set_title('Model Performance Comparison', fontweight='bold')
axes[0,0].set_xticks(x + width * 2)
axes[0,0].set_xticklabels(model_names, rotation=15)
axes[0,0].legend(bbox_to_anchor=(1.05, 1), loc='upper left')
axes[0,0].grid(True, alpha=0.3)

# 2. ROC Curves
for name, result in model_results.items():
    try:
        fpr, tpr, _ = roc_curve(y_test, result['probabilities'])
        axes[0,1].plot(fpr, tpr, label=f"{name} (AUC = {result['auc_score']:.3f})", linewidth=2)
    except:
        continue

axes[0,1].plot([0, 1], [0, 1], 'k--', label='Random', linewidth=1)
axes[0,1].set_xlabel('False Positive Rate')
axes[0,1].set_ylabel('True Positive Rate')
axes[0,1].set_title('ROC Curves', fontweight='bold')
axes[0,1].legend()
axes[0,1].grid(True, alpha=0.3)

# 3. Precision-Recall Curves
for name, result in model_results.items():
    try:
        precision, recall, _ = precision_recall_curve(y_test, result['probabilities'])
        axes[0,2].plot(recall, precision, label=f"{name}", linewidth=2)
    except:
        continue

# Baseline (random)
baseline_precision = y_test.mean()
axes[0,2].axhline(y=baseline_precision, color='k', linestyle='--', label=f'Baseline ({baseline_precision:.3f})')
axes[0,2].set_xlabel('Recall')
axes[0,2].set_ylabel('Precision')
axes[0,2].set_title('Precision-Recall Curves', fontweight='bold')
axes[0,2].legend()
axes[0,2].grid(True, alpha=0.3)

# 4. Confusion Matrix for best model
best_result = model_results[best_model_name]
cm = confusion_matrix(y_test, best_result['predictions'])

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1,0],
           xticklabels=['Normal', 'Fraud'], yticklabels=['Normal', 'Fraud'])
axes[1,0].set_title(f'Confusion Matrix - {best_model_name}', fontweight='bold')
axes[1,0].set_xlabel('Predicted')
axes[1,0].set_ylabel('Actual')

# 5. Feature Importance (for Random Forest)
if 'Random Forest' in model_results:
    rf_model = trained_models['Random Forest']
    feature_importance = rf_model.feature_importances_
    feature_names = X.columns
    
    # Get top 10 features
    top_indices = np.argsort(feature_importance)[-10:]
    top_features = [feature_names[i] for i in top_indices]
    top_importance = feature_importance[top_indices]
    
    axes[1,1].barh(top_features, top_importance, color='skyblue')
    axes[1,1].set_title('Top 10 Feature Importance (Random Forest)', fontweight='bold')
    axes[1,1].set_xlabel('Importance')
else:
    axes[1,1].text(0.5, 0.5, 'Feature Importance\n(Random Forest not available)', 
                   ha='center', va='center', transform=axes[1,1].transAxes)
    axes[1,1].set_title('Feature Importance', fontweight='bold')

# 6. Prediction Distribution
fraud_probs = best_result['probabilities'][y_test == 1]
normal_probs = best_result['probabilities'][y_test == 0]

axes[1,2].hist(normal_probs, bins=50, alpha=0.7, label='Normal', color='blue', density=True)
axes[1,2].hist(fraud_probs, bins=50, alpha=0.7, label='Fraud', color='red', density=True)
axes[1,2].set_xlabel('Fraud Probability')
axes[1,2].set_ylabel('Density')
axes[1,2].set_title(f'Prediction Distribution - {best_model_name}', fontweight='bold')
axes[1,2].legend()
axes[1,2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Detailed classification report for best model
print(f"\n📋 DETAILED CLASSIFICATION REPORT - {best_model_name}")
print("="*60)
print(classification_report(y_test, best_result['predictions'], 
                          target_names=['Normal', 'Fraud'], digits=3))

print("\n✅ Model evaluation completed successfully!")

## 💼 Business Insights and Recommendations

Generate actionable business insights and recommendations based on the fraud detection analysis.

In [ ]:
print("💼 BUSINESS INSIGHTS AND RECOMMENDATIONS")
print("="*80)

# Financial impact analysis
total_fraud_amount = df[df['is_fraud'] == 1]['amount'].sum()
avg_fraud_amount = df[df['is_fraud'] == 1]['amount'].mean()
total_normal_amount = df[df['is_fraud'] == 0]['amount'].sum()
avg_normal_amount = df[df['is_fraud'] == 0]['amount'].mean()

# Best model performance
best_result = model_results[best_model_name]
detected_frauds = int(best_result['recall'] * y_test.sum())
missed_frauds = y_test.sum() - detected_frauds
prevented_loss = detected_frauds * avg_fraud_amount
potential_loss = missed_frauds * avg_fraud_amount

print(f"\n🏆 BEST PERFORMING MODEL: {best_model_name}")
print(f"   • F1-Score: {best_result['f1_score']:.3f}")
print(f"   • Precision: {best_result['precision']:.3f}")
print(f"   • Recall: {best_result['recall']:.3f}")
print(f"   • AUC: {best_result['auc_score']:.3f}")

print(f"\n💰 FINANCIAL IMPACT ANALYSIS:")
print(f"   • Total fraud amount in dataset: ₹{total_fraud_amount:,.2f}")
print(f"   • Average fraud transaction: ₹{avg_fraud_amount:,.2f}")
print(f"   • Average normal transaction: ₹{avg_normal_amount:,.2f}")
print(f"   • Fraud transactions are {avg_fraud_amount/avg_normal_amount:.1f}x larger on average")

print(f"\n🎯 MODEL IMPACT ON TEST SET:")
print(f"   • Total test fraud cases: {y_test.sum()}")
print(f"   • Frauds detected by model: {detected_frauds} ({best_result['recall']*100:.1f}%)")
print(f"   • Frauds missed by model: {missed_frauds} ({(1-best_result['recall'])*100:.1f}%)")
print(f"   • Estimated loss prevented: ₹{prevented_loss:,.2f}")
print(f"   • Potential remaining loss: ₹{potential_loss:,.2f}")

# Risk pattern analysis
fraud_by_category = df.groupby('merchant_category')['is_fraud'].agg(['count', 'sum', 'mean']).round(4)
fraud_by_category.columns = ['total_txns', 'fraud_count', 'fraud_rate']
fraud_by_category = fraud_by_category.sort_values('fraud_rate', ascending=False)

fraud_by_hour = df.groupby('hour_of_day')['is_fraud'].mean().sort_values(ascending=False)
high_risk_hours = fraud_by_hour[fraud_by_hour > fraud_by_hour.mean() + fraud_by_hour.std()]

print(f"\n🚨 HIGH-RISK PATTERNS IDENTIFIED:")
print(f"\n   📊 Top Risk Merchant Categories:")
for i, (category, data) in enumerate(fraud_by_category.head(3).iterrows()):
    print(f"      {i+1}. {category}: {data['fraud_rate']*100:.1f}% fraud rate ({data['fraud_count']} out of {data['total_txns']} transactions)")

print(f"\n   🕐 High-Risk Hours:")
if len(high_risk_hours) > 0:
    for hour, rate in high_risk_hours.head(3).items():
        print(f"      • Hour {hour:02d}:00: {rate*100:.1f}% fraud rate")
else:
    print(f"      • No significantly high-risk hours identified")

# Amount-based analysis
high_amount_threshold = df['amount'].quantile(0.95)
high_amount_fraud_rate = df[df['amount'] > high_amount_threshold]['is_fraud'].mean()
city_mismatch_fraud_rate = df[df['city_mismatch'] == True]['is_fraud'].mean()
weekend_fraud_rate = df[df['is_weekend'] == True]['is_fraud'].mean()
weekday_fraud_rate = df[df['is_weekend'] == False]['is_fraud'].mean()

print(f"\n   💸 Amount-Based Risks:")
print(f"      • High-value transactions (>₹{high_amount_threshold:,.0f}): {high_amount_fraud_rate*100:.1f}% fraud rate")
print(f"      • City mismatch transactions: {city_mismatch_fraud_rate*100:.1f}% fraud rate")
print(f"      • Weekend vs Weekday fraud: {weekend_fraud_rate*100:.1f}% vs {weekday_fraud_rate*100:.1f}%")

# Generate recommendations
false_positive_rate = 1 - best_result['precision']
expected_daily_alerts = int(len(df) * best_result['precision'] / 90)  # Assuming 90-day dataset

print(f"\n📈 RECOMMENDATIONS FOR BANKS:")
print(f"\n   🔧 Technical Implementation:")
print(f"      1. Deploy {best_model_name} for real-time fraud detection")
print(f"      2. Set fraud probability threshold based on business requirements")
print(f"      3. Implement model monitoring and regular retraining (monthly)")
print(f"      4. Create fallback rules for high-risk patterns")

print(f"\n   🎯 Risk-Based Rules:")
print(f"      1. Enhanced monitoring for {', '.join(fraud_by_category.head(2).index)} transactions")
print(f"      2. Step-up authentication for transactions >₹{high_amount_threshold:,.0f}")
print(f"      3. Additional verification for city mismatch transactions")
print(f"      4. Increased scrutiny during high-risk hours: {list(high_risk_hours.head(3).index)}")

print(f"\n   📊 Operational Metrics:")
print(f"      • Expected daily fraud alerts: ~{expected_daily_alerts}")
print(f"      • False positive rate: {false_positive_rate:.1%} (review process needed)")
print(f"      • Customer friction: Moderate (due to false positives)")
print(f"      • Manual review capacity needed: {expected_daily_alerts * 2} cases/day")

print(f"\n   💡 Strategic Initiatives:")
print(f"      1. Customer education on fraud prevention")
print(f"      2. Merchant risk scoring program")
print(f"      3. Real-time transaction monitoring dashboard")
print(f"      4. Integration with external fraud databases")
print(f"      5. Mobile app notifications for suspicious transactions")

print(f"\n⚠️ IMPORTANT CONSIDERATIONS:")
print(f"   • This model is trained on synthetic data - validate with real data")
print(f"   • Consider regulatory compliance requirements (PCI DSS, etc.)")
print(f"   • Implement proper data privacy and security measures")
print(f"   • Monitor for model drift and adversarial attacks")
print(f"   • Maintain customer experience while preventing fraud")

print(f"\n✅ Business analysis completed successfully!")

## 🔍 Model Interpretation and Feature Analysis

Deep dive into model behavior and feature importance for better understanding.

In [ ]:
print("🔍 MODEL INTERPRETATION AND FEATURE ANALYSIS")
print("="*60)

# Feature importance analysis (for Random Forest)
if 'Random Forest' in trained_models:
    rf_model = trained_models['Random Forest']
    feature_importance = rf_model.feature_importances_
    feature_names = X.columns
    
    # Create feature importance dataframe
    importance_df = pd.DataFrame({
        'feature': feature_names,
        'importance': feature_importance
    }).sort_values('importance', ascending=False)
    
    print(f"\n🏆 TOP 15 MOST IMPORTANT FEATURES (Random Forest):")
    print("-" * 50)
    for i, (_, row) in enumerate(importance_df.head(15).iterrows()):
        print(f"{i+1:2d}. {row['feature']:<30} {row['importance']:.4f}")
    
    # Interactive feature importance plot
    fig_importance = px.bar(importance_df.head(15), 
                           x='importance', y='feature',
                           orientation='h',
                           title='Top 15 Feature Importance (Random Forest)',
                           labels={'importance': 'Feature Importance', 'feature': 'Features'})
    fig_importance.update_layout(height=600, yaxis={'categoryorder':'total ascending'})
    fig_importance.show()

# Analyze model predictions by different segments
print(f"\n📊 MODEL PERFORMANCE BY SEGMENTS:")
print("-" * 40)

# Performance by amount ranges
test_df = X_test.copy()
test_df['actual_fraud'] = y_test
test_df['predicted_fraud'] = best_result['predictions']
test_df['fraud_probability'] = best_result['probabilities']

# Add amount ranges
test_df['amount_range'] = pd.cut(test_df['amount'], bins=5, labels=['Very Low', 'Low', 'Medium', 'High', 'Very High'])

# Performance by amount range
performance_by_amount = test_df.groupby('amount_range').apply(
    lambda x: pd.Series({
        'total_transactions': len(x),
        'actual_frauds': x['actual_fraud'].sum(),
        'detected_frauds': (x['actual_fraud'] * x['predicted_fraud']).sum(),
        'false_positives': ((1-x['actual_fraud']) * x['predicted_fraud']).sum(),
        'precision': (x['actual_fraud'] * x['predicted_fraud']).sum() / (x['predicted_fraud'].sum() + 1e-8),
        'recall': (x['actual_fraud'] * x['predicted_fraud']).sum() / (x['actual_fraud'].sum() + 1e-8)
    })
).round(3)

print(f"\n💰 Performance by Amount Range:")
print(performance_by_amount)

# Create threshold analysis
thresholds = np.arange(0.1, 1.0, 0.1)
threshold_analysis = []

for threshold in thresholds:
    y_pred_thresh = (best_result['probabilities'] >= threshold).astype(int)
    
    precision = (y_pred_thresh * y_test).sum() / (y_pred_thresh.sum() + 1e-8)
    recall = (y_pred_thresh * y_test).sum() / (y_test.sum() + 1e-8)
    f1 = 2 * (precision * recall) / (precision + recall + 1e-8)
    
    threshold_analysis.append({
        'threshold': threshold,
        'precision': precision,
        'recall': recall,
        'f1_score': f1,
        'predicted_frauds': y_pred_thresh.sum()
    })

threshold_df = pd.DataFrame(threshold_analysis)

# Plot threshold analysis
fig_threshold = go.Figure()

fig_threshold.add_trace(go.Scatter(x=threshold_df['threshold'], y=threshold_df['precision'],
                                  mode='lines+markers', name='Precision', line=dict(color='blue')))
fig_threshold.add_trace(go.Scatter(x=threshold_df['threshold'], y=threshold_df['recall'],
                                  mode='lines+markers', name='Recall', line=dict(color='red')))
fig_threshold.add_trace(go.Scatter(x=threshold_df['threshold'], y=threshold_df['f1_score'],
                                  mode='lines+markers', name='F1-Score', line=dict(color='green')))

fig_threshold.update_layout(
    title=f'Threshold Analysis - {best_model_name}',
    xaxis_title='Fraud Probability Threshold',
    yaxis_title='Score',
    height=500
)
fig_threshold.show()

# Find optimal threshold
optimal_threshold_idx = threshold_df['f1_score'].idxmax()
optimal_threshold = threshold_df.loc[optimal_threshold_idx, 'threshold']
optimal_f1 = threshold_df.loc[optimal_threshold_idx, 'f1_score']

print(f"\n🎯 OPTIMAL THRESHOLD ANALYSIS:")
print(f"   • Optimal threshold: {optimal_threshold:.1f}")
print(f"   • F1-Score at optimal threshold: {optimal_f1:.3f}")
print(f"   • Precision at optimal threshold: {threshold_df.loc[optimal_threshold_idx, 'precision']:.3f}")
print(f"   • Recall at optimal threshold: {threshold_df.loc[optimal_threshold_idx, 'recall']:.3f}")

print(f"\n📋 THRESHOLD RECOMMENDATIONS:")
print(f"   • Conservative (Low False Positives): Use threshold 0.7-0.8")
print(f"   • Balanced (Optimal F1): Use threshold {optimal_threshold:.1f}")
print(f"   • Aggressive (Catch More Frauds): Use threshold 0.2-0.3")

print(f"\n✅ Model interpretation completed successfully!")

## 💾 Export Results and Model

Save the trained model and results for future use.

In [ ]:
print("💾 EXPORTING RESULTS AND MODEL")
print("="*50)

# Create results directory
results_dir = 'fraud_detection_results'
os.makedirs(results_dir, exist_ok=True)

try:
    # 1. Save model performance summary
    performance_summary = pd.DataFrame.from_dict(
        {name: {k: v for k, v in result.items() 
               if k not in ['model', 'predictions', 'probabilities']}
         for name, result in model_results.items()}, 
        orient='index'
    )
    performance_summary.to_csv(f'{results_dir}/model_performance_summary.csv')
    print(f"✅ Model performance summary saved to {results_dir}/model_performance_summary.csv")
    
    # 2. Save predictions for best model
    predictions_df = pd.DataFrame({
        'actual_fraud': y_test.values,
        'predicted_fraud': best_result['predictions'],
        'fraud_probability': best_result['probabilities'],
        'correct_prediction': (y_test.values == best_result['predictions'])
    })
    predictions_df.to_csv(f'{results_dir}/best_model_predictions.csv', index=False)
    print(f"✅ Best model predictions saved to {results_dir}/best_model_predictions.csv")
    
    # 3. Save feature importance (if available)
    if 'Random Forest' in trained_models:
        importance_df.to_csv(f'{results_dir}/feature_importance.csv', index=False)
        print(f"✅ Feature importance saved to {results_dir}/feature_importance.csv")
    
    # 4. Save threshold analysis
    threshold_df.to_csv(f'{results_dir}/threshold_analysis.csv', index=False)
    print(f"✅ Threshold analysis saved to {results_dir}/threshold_analysis.csv")
    
    # 5. Create summary report
    with open(f'{results_dir}/fraud_detection_report.txt', 'w') as f:
        f.write("CREDIT CARD FRAUD DETECTION - ANALYSIS REPORT\n")
        f.write("=" * 50 + "\n\n")
        f.write(f"Analysis Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"Dataset Size: {len(df):,} transactions\n")
        f.write(f"Fraud Rate: {df['is_fraud'].mean()*100:.3f}%\n\n")
        
        f.write("BEST MODEL PERFORMANCE:\n")
        f.write(f"Model: {best_model_name}\n")
        f.write(f"F1-Score: {best_result['f1_score']:.3f}\n")
        f.write(f"Precision: {best_result['precision']:.3f}\n")
        f.write(f"Recall: {best_result['recall']:.3f}\n")
        f.write(f"AUC: {best_result['auc_score']:.3f}\n\n")
        
        f.write("KEY RECOMMENDATIONS:\n")
        f.write(f"1. Deploy {best_model_name} for real-time fraud detection\n")
        f.write(f"2. Use optimal threshold: {optimal_threshold:.1f}\n")
        f.write(f"3. Monitor high-risk merchant categories\n")
        f.write(f"4. Implement regular model retraining\n")
    
    print(f"✅ Summary report saved to {results_dir}/fraud_detection_report.txt")
    
    # 6. Display final summary
    print(f"\n📊 FINAL SUMMARY")
    print(f"=" * 40)
    print(f"📁 Results exported to: {results_dir}/")
    print(f"🏆 Best Model: {best_model_name}")
    print(f"🎯 F1-Score: {best_result['f1_score']:.3f}")
    print(f"⚡ Recall: {best_result['recall']:.3f} (fraud detection rate)")
    print(f"🎪 Precision: {best_result['precision']:.3f} (accuracy of fraud alerts)")
    
    print(f"\n🚀 NEXT STEPS FOR IMPLEMENTATION:")
    print(f"   1. Validate model with real transaction data")
    print(f"   2. Set up real-time prediction infrastructure")
    print(f"   3. Implement monitoring and alerting system")
    print(f"   4. Create customer notification workflows")
    print(f"   5. Establish model retraining schedule")

except Exception as e:
    print(f"❌ Error exporting results: {str(e)}")

print(f"\n✅ Export completed successfully!")

## 📚 Instructions and Guidelines

### 🎓 For Students:

#### **Learning Objectives Achieved:**
1. **Data Analysis**: Understanding fraud patterns in transaction data
2. **Machine Learning**: Training and evaluating multiple ML models
3. **Model Comparison**: Comparing different algorithms for fraud detection
4. **Business Application**: Translating ML results into business insights
5. **Performance Optimization**: Finding optimal thresholds and parameters

#### **Experiments to Try:**
1. **Different Models**: Try XGBoost, SVM, or Neural Networks
2. **Feature Engineering**: Create new features (transaction velocity, merchant history)
3. **Sampling Techniques**: Compare SMOTE with other balancing methods
4. **Threshold Optimization**: Find business-optimal thresholds
5. **Cross-Validation**: Implement k-fold cross-validation

#### **Advanced Topics:**
- **Ensemble Methods**: Combine multiple models
- **Deep Learning**: Use neural networks for sequence modeling
- **Online Learning**: Implement models that adapt over time
- **Explainable AI**: Use SHAP values for model interpretation

### 🏦 For Banks:

#### **Implementation Roadmap:**
1. **Proof of Concept** (1-2 months)
   - Validate model with historical data
   - Test with small transaction subset
   - Measure baseline performance

2. **Pilot Deployment** (2-3 months)
   - Deploy in shadow mode
   - Compare with existing systems
   - Collect feedback from fraud analysts

3. **Full Production** (3-6 months)
   - Real-time implementation
   - Integration with existing systems
   - Staff training and procedures

#### **Technical Requirements:**
- **Infrastructure**: Real-time scoring capability (<100ms)
- **Data Pipeline**: Stream processing for live transactions
- **Monitoring**: Model performance and drift detection
- **Compliance**: PCI DSS, data privacy regulations

#### **Risk Management:**
- **Model Risk**: Regular validation and backtesting
- **Operational Risk**: Fallback systems and manual processes
- **Regulatory Risk**: Compliance with financial regulations
- **Reputational Risk**: Customer experience considerations

### ⚠️ Important Considerations:

#### **Data Quality:**
- This notebook uses synthetic data for demonstration
- Real-world data may have different patterns and challenges
- Always validate models with production data before deployment

#### **Ethical Considerations:**
- Ensure fair treatment across customer segments
- Monitor for bias in model predictions
- Maintain transparency in automated decisions

#### **Continuous Improvement:**
- Regular model retraining (monthly/quarterly)
- Monitor for fraud pattern evolution
- Incorporate feedback from fraud investigations

---

**🎉 Congratulations!** You have successfully completed the Credit Card Fraud Detection analysis. This comprehensive notebook provides a solid foundation for understanding and implementing ML-based fraud detection systems in real-world financial environments.